# E2 Exercises: Feature Scaling

> 1,200 admission blood panels · ~90 minutes · needs `notebooks/C2_Feature_Scaling.ipynb`

## Objective → task map

Every objective the chapter states is assessed by at least one task below. Where the
chapter's own embedded exercises already cover one, that is said here rather than
duplicated.

| # | Objective | Assessed by |
|---|---|---|
| 1 | Choose between StandardScaler, MinMaxScaler and RobustScaler | Tasks 4, 5 |
| 2 | Fit on the training set and transform both without leaking | Tasks 2, 5 |
| 3 | Show scaling's effect on a distance model and on a tree | Task 3 |
| 4 | Recognise when a heavy tail makes StandardScaler wrong | Tasks 4, 5 |
| 5 | Apply a log transform and say what it does to a skewed column | Task 5 |
| 6 | Recognise when scaling is needed at all | Tasks 1, 3 |

## Before you start

**Everything here is solvable with C2 alone.** `StandardScaler`, `MinMaxScaler`, `RobustScaler`, `np.log1p`, `train_test_split`, `KNeighborsClassifier`, `RandomForestClassifier`, `Pipeline` and `.describe()`. Task 5 uses `ColumnTransformer` and `FunctionTransformer` to apply a log to two columns only; both are demonstrated in C6, and the code is given in the brief if you need it.

Nothing below needs a method the chapter did not cover. Where a task is hard, it is
hard because of a decision you have to make and justify, not because of syntax.

**Marks.** 100 in total. Task 5 carries 45 and is the only one that asks you to recommend a deployment.

**Dataset path used in this notebook:** `../data/clinic_labs.csv`

---

## Task 1: Eight columns, eight scales

**Warm-up · 10 marks · Recognise when scaling is needed**

Load `clinic_labs.csv`. For the eight feature columns (everything except
`patient_id` and `sepsis_flag`), report the mean, standard deviation, min and max.

Then answer:

**(a)** Which column has the largest standard deviation, and how many times larger is
it than the smallest?

**(b)** A KNN classifier computes Euclidean distance across all eight columns as they
stand. Roughly what share of that distance comes from your answer to (a)? A one-line
estimate is enough; you are being asked for the order of magnitude, not a proof.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

pd.set_option('display.max_columns', None)
df = pd.read_csv('../../00-data/clinic_labs.csv')

FEATURES = ['age_years', 'temperature_c', 'wbc_count', 'serum_sodium',
            'systolic_bp', 'platelet_count', 'creatinine_umol_l',
            'troponin_ng_l']

# Your code here

**Your answer to Task 1:**

*(write here)*

---

## Task 2: Fit on train, transform both

**Warm-up · 15 marks · Fit a scaler without leaking**

Split the data (`test_size=0.25, random_state=42, stratify=y`). Fit a `StandardScaler`
**on the training set only** and transform both sets.

Report, for the scaled training set and the scaled test set separately, the mean and
standard deviation of `serum_sodium`.

Then answer: **the training column has mean 0 and std 1 exactly. The test column does
not. Is that a bug? Answer in two sentences.**

In [ ]:
# Your code here

**Your answer to Task 2:**

*(write here)*

---

## Task 3: Who cares about scaling, and who does not

**Core · 15 marks · Show the effect of scaling on a distance model and on a tree**

Using the split from Task 2, fit and score four models on the test set:

1. `KNeighborsClassifier(15)` on the raw features
2. `KNeighborsClassifier(15)` on the scaled features
3. `RandomForestClassifier(200, random_state=42)` on the raw features
4. the same forest on the scaled features

Report all four accuracies in one table.

Then answer: **one pair moved a lot and one barely moved. Explain both results in terms
of what each algorithm does with a column's values.**

In [ ]:
# Your code here

**Your answer to Task 3:**

*(write here)*

---

## Task 4: Three scalers on a column with a tail

**Core · 15 marks · Recognise when a heavy tail makes StandardScaler the wrong choice**

`troponin_ng_l` is right-skewed with a long tail of genuine cardiac cases.

Apply `StandardScaler`, `MinMaxScaler` and `RobustScaler` to **all eight feature
columns** and report, for each scaler, the interquartile range of every scaled column.

Plot the three scaled versions of `troponin_ng_l` side by side as histograms.

Then recommend one for use with a KNN model, in one sentence, with the reason.

> A warning about the obvious metric. "What share of the output range does the middle 50%
> occupy?" cannot distinguish these three at all: every one of them is a linear map, and a
> linear map multiplies the IQR and the range by the same constant. That ratio comes out
> identical to four decimal places for all three. The question worth asking is not what a
> scaler does to one column, it is whether the columns end up comparable **with each
> other**, which is the only thing a distance calculation cares about.

In [ ]:
# Your code here

**Your answer to Task 4:**

*(write here)*

---

## Task 5: "The scaler never sees y, so it cannot leak"

**Judgement · 45 marks · Choose a scaler, and defend the order of operations**

A colleague sends you their sepsis model and this justification:

> *"I scale the whole dataset before splitting. It's fine, `StandardScaler` only reads
> X, it never touches y. A transformation that doesn't use the labels can't leak. And
> doing it once up front is cleaner than fitting on train and remembering to transform
> the test set separately."*

**Part 1 (10 marks).** Reproduce their pipeline exactly (scale everything, then split,
then fit `KNeighborsClassifier(15)`) and report the test accuracy. Then run the correct
version and report that. State the difference.

Then repeat both, but training on a random sample of only 60 patients instead of 900.
Report the difference again. Say what changed and why.

**Part 2 (10 marks).** Their argument has one true premise and one false conclusion.
Say which is which. Your answer must name **what specifically** crosses from the test set
into the training process, given that `y` genuinely never enters the scaler.

**Part 3 (10 marks).** Build and honestly evaluate three candidate pipelines:

- **A** StandardScaler → KNN
- **B** RobustScaler → KNN
- **C** `np.log1p` on `troponin_ng_l` and `creatinine_umol_l`, then StandardScaler → KNN

Report all three. Then say whether the log transform belongs **before** or **after** the
scaler, and why the other order does not work.

**Part 4 (15 marks).** Northbridge will deploy one of A, B or C. Recommend one and defend
it in a paragraph. Your answer must address what happens when a patient arrives whose
troponin is higher than anything in the training set, because on a cardiac ward that will
happen in the first week.

In [ ]:
# Your code here

**Your answer to Task 5:**

*(write here)*

---

## Before you submit

- [ ] Every scaler in this notebook was fitted on training rows only
- [ ] Your Task 3 answer explains the forest result through the split rule
- [ ] Task 5 Part 2 names `scaler.mean_` and `scaler.scale_`, not just "information"
- [ ] Task 5 Part 4 argues from the out-of-range patient, not from the accuracy column
- [ ] You can state which of your models would be unaffected if you deleted every scaler